# Day 07 — An evaluation baseline: labelled questions, retrieval metrics, every error, measured thresholds

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-07/LEARN.md`](../docs/day-07/LEARN.md).*

Days 01–06 built a grounded RAG pipeline and tried it on a few dozen questions, one at a time. Every judgement so far ("BM25 misses this one", "0.60 seems a safe threshold") came from looking at examples. Before Day 08 changes anything, we need a way to say whether a change made things **better or worse**, on the same questions and the same documents, with numbers anyone can reproduce.

That takes four things, and this notebook builds each one:

1. **A labelled set** ([`evals/questions-v1.jsonl`](../evals/questions-v1.jsonl)): 81 queries, each with who asks it, the evidence that answers it, and the chunk ids that hold that evidence, tied to a recorded snapshot of the documents.
2. **Metrics** that need those labels: hit rate@k, recall@k and MRR, for BM25 and the dense retriever, on the same queries.
3. **An error log**: every query where the answer isn't in the top 3, with a diagnosis written by hand.
4. **Measured thresholds**: Day 06's abstention settings, chosen by eye, measured on questions it hadn't seen.

Everything runs locally: no API key, no new packages, about 15 seconds.

## 0. Setup

No new packages. The shared code in [`src/rag_from_scratch/`](../src/rag_from_scratch/) gains **Day 06's answering pipeline** (`answering.py`: retrieve as a user, the gate, the context, the extractive reader, the citation check), because today measures it. Its thresholds are now arguments with Day 06's values as defaults, so we can measure them without editing anything. `tests/test_shared_code.py` checks that the module behaves exactly like the Day 06 notebook, with Day 06's settings and with others.

In [1]:
import hashlib
import json
import math
import platform
import re
import sys
import time
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
# huggingface_hub's progress bars warn when Jupyter widgets aren't installed; they still work as plain text.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import numpy as np
    import onnxruntime as ort
except ImportError as error:
    raise ImportError("Day 07 uses Day 04's packages (numpy, onnxruntime, tokenizers, huggingface-hub). From the "
                      "repository root run: python -m pip install -r requirements.txt") from error


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))           # import the shared code without installing it

from rag_from_scratch.answering import (DIRECTORY, MIN_RETRIEVAL_SCORE, MIN_SENTENCE_SCORE,  # Day 06's pipeline
                                        ExtractiveReader, Pipeline, assemble, unknown_names)  # (new in src/)
from rag_from_scratch.bm25 import BM25Index                           # Day 01's BM25
from rag_from_scratch.chunking import section_chunks                  # Day 03's chunker
from rag_from_scratch.corpus import corpus_after                      # Day 05's change scenarios
from rag_from_scratch.embeddings import Embedder                      # Day 04's model
from rag_from_scratch.index import VectorIndex, sync                  # Day 05's index
from rag_from_scratch.ingest import ingest                            # Day 02's pipeline

print(f"Python {platform.python_version()} | numpy {np.__version__} | onnxruntime {ort.__version__}")

Python 3.13.16 | numpy 2.4.6 | onnxruntime 1.31.0


In [2]:
TOP_K = 3                # what Day 06's pipeline reads: a query is an ERROR if no relevant chunk is in its top 3
MAX_K = 10               # how deep the metrics look (hit@10, MRR@10)
K_VALUES = (1, 3, 5, 10)
EVALS = REPO_ROOT / "evals"                                      # versioned with the code: labels, manifest, baselines
QUESTIONS_FILE = EVALS / "questions-v1.jsonl"
MANIFEST_FILE = EVALS / "manifest.json"
BASELINE_FILE = EVALS / "baselines" / "day-07-retrieval.json"    # the accepted baseline, committed and reviewed
OUT_DIR = REPO_ROOT / "outputs" / "day-07"                       # this run's results; outputs/ is git-ignored

## 1. The labelled set

A **labelled query** is a question plus the ground truth needed to grade a retriever's answer to it. Ours has three parts that matter:

- **who asks** (`user`): alice is an employee, maria a manager. Day 05 and 06 made retrieval permission-aware, so the right answer depends on the asker. Alice's pay question has an answer in the documents, but not one *she* may read: for her the correct behaviour is to find nothing.
- **the evidence** (`evidence`): the exact strings that must appear in a chunk for it to answer the question, written by hand (Day 03's probes work the same way).
- **the label** (`answer_chunks`): the chunks this user may read that contain all of the evidence. It isn't typed by hand: it was **derived from the evidence** on today's documents and then frozen in the file. `restricted_chunks` records evidence the user may not read.

Where the 81 queries come from:

| Origin | Queries | Why they're in |
|---|---|---|
| Day 03's probes | 13 | each tests one way chunking can fail (table rows, headings, two sections) |
| Day 01's failures | 4 | vocabulary mismatch, stemming, two out-of-scope questions |
| Day 06's check list | 7 | permissions (the pay question for maria and alice) and five unanswerable questions |
| Day 04's lookups | 30 | an exact amount, time, address or link typed as the query; run as maria, who reads all 44 chunks, like Day 04 |
| **new today** | 27 | 15 answerable, 12 not, phrased the way people ask; written **before any retriever was run on them** |

The **split** matters as much as the labels. The first 54 were seen while Days 01–06 were built, and Day 06 chose its thresholds by eye on some of them: they are the **dev** split, the ones we may tune on. Today's 27 are the **test** split: we report on them and never tune on them. Otherwise we'd be grading the pipeline on the questions it was fitted to.

In [3]:
@dataclass(frozen=True)
class Query:
    id: str
    question: str
    user: str                            # who asks: the right answer depends on what this person may read
    kind: str                            # "question", or "lookup" (an exact value typed as the query, Day 04)
    split: str                           # "dev" (seen while building Days 01-06) or "test" (new today)
    origin: str                          # where it came from: day01-failure, day03-probe, day04-lookup, ...
    tags: tuple[str, ...]
    evidence: tuple[str, ...]            # strings that must ALL appear in a chunk for it to answer the question
    facts: tuple[str, ...]               # strings a correct answer contains (for answer checks, Day 12)
    answer_chunks: tuple[str, ...]       # THE LABEL: chunks this user may read that hold the evidence
    restricted_chunks: tuple[str, ...]   # chunks that hold the evidence but this user may NOT read
    note: str

    @property
    def answerable(self) -> bool:
        return bool(self.answer_chunks)


def load_queries(path: Path) -> list[Query]:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return [Query(**{k: tuple(v) if isinstance(v, list) else v for k, v in row.items()}) for row in rows]


manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
queries = load_queries(QUESTIONS_FILE)
by_qid = {q.id: q for q in queries}
file_sha256 = hashlib.sha256(QUESTIONS_FILE.read_bytes()).hexdigest()

print(f"{len(queries)} labelled queries in {QUESTIONS_FILE.relative_to(REPO_ROOT).as_posix()} "
      f"(version {manifest['version']}, sha256 {file_sha256[:12]}...)\n")
print(f"{'':<28}{'dev':>5}{'test':>6}{'total':>7}")
for label, belongs in [("questions, answerable", lambda q: q.kind == "question" and q.answerable),
                       ("questions, not answerable", lambda q: q.kind == "question" and not q.answerable),
                       ("lookups", lambda q: q.kind == "lookup")]:
    dev, test = (sum(belongs(q) and q.split == split for q in queries) for split in ("dev", "test"))
    print(f"{label:<28}{dev:>5}{test:>6}{dev + test:>7}")
print("\nby origin: " + ", ".join(f"{origin} {n}" for origin, n in Counter(q.origin for q in queries).items()))
print("asked by: " + ", ".join(f"{user} {n}" for user, n in Counter(q.user for q in queries).items()))


def show_query(q: Query) -> None:
    quoted = lambda values: ", ".join(f'"{v}"' for v in values) or "-"
    print(f"\n{q.id} | {q.user} | {q.kind} | {q.split} | {q.origin}")
    print(f"  question:          {q.question}")
    print(f"  evidence:          {quoted(q.evidence)}")
    print(f"  answer_chunks:     {', '.join(q.answer_chunks) or '-'}")
    print(f"  restricted_chunks: {', '.join(q.restricted_chunks) or '-'}")
    print(f"  facts:             {quoted(q.facts)}")
    print(f"  note:              {q.note}")


for qid in ("d03-01", "d06-02", "d07-u03"):
    show_query(by_qid[qid])

81 labelled queries in evals/questions-v1.jsonl (version 1, sha256 56d356f9cf8e...)

                              dev  test  total
questions, answerable          16    15     31
questions, not answerable       8    12     20
lookups                        30     0     30

by origin: day03-probe 13, day01-failure 4, day06-check 7, day04-lookup 30, day07-new 27
asked by: alice 49, maria 32

d03-01 | alice | question | dev | day03-probe
  question:          What is the hotel limit per night in Singapore?
  evidence:          "Singapore | USD 220", "Hotel per night"
  answer_chunks:     travel-policy#daily-limits-by-city
  restricted_chunks: -
  facts:             "220"
  note:              Day 03: PDF table: row + column header

d06-02 | alice | question | dev | day06-check
  question:          When do pay rises take effect?
  evidence:          "take effect from 1 April"
  answer_chunks:     -
  restricted_chunks: manager-pay-review-guide#overview
  facts:             -
  note:         

### 1.1 Labels belong to a snapshot

A label like "the answer is in `password-and-mfa-standard#passwords`" is only true for one version of the documents. So [`evals/manifest.json`](../evals/manifest.json) records the **snapshot** the labels were written against: every chunk's text hash and who may read it. Before measuring anything, `validate()` checks that the documents still match the snapshot and derives every label again from its evidence. If anything moved, it says which chunks and which labels to review, and the notebook stops: a metric computed on stale labels is wrong without looking wrong.

In [4]:
def holds(evidence, text: str) -> bool:
    # Day 04's whole-match rule: every evidence string, never as part of a longer number ("USD 25" isn't in "USD 250")
    return bool(evidence) and all(re.search(rf"(?<!\w){re.escape(e)}(?!\w|[,.]\d)", text) for e in evidence)


def snapshot(chunks) -> dict:
    # What the labels depend on: every chunk's text (as its hash) and who may read it
    return {c.chunk_id: {"hash": c.chunk_hash, "groups": list(c.allowed_groups)} for c in chunks}


def fingerprint(snap: dict) -> str:
    return hashlib.sha256(json.dumps(snap, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()


def validate(queries, chunks, manifest, directory=DIRECTORY) -> list[str]:
    problems = []
    if len({q.id for q in queries}) != len(queries) or len({(q.user, q.question) for q in queries}) != len(queries):
        problems.append("duplicate ids, or the same question twice for the same user")
    if {user: sorted(groups) for user, groups in directory.items()} != manifest["directory"]:
        problems.append("the user directory changed since the labels were written")
    # 1. The snapshot: every chunk's text hash and permissions, against the ones recorded with the labels
    now, then = snapshot(chunks), manifest["snapshot"]["chunks"]
    for cid in sorted(set(now) | set(then)):
        if cid not in now:
            problems.append(f"chunk {cid}: gone")
        elif cid not in then:
            problems.append(f"chunk {cid}: new")
        elif now[cid] != then[cid]:
            problems.append(f"chunk {cid}: {'text' if now[cid]['hash'] != then[cid]['hash'] else 'permissions'} changed")
    # 2. Every label, derived again from its evidence on today's chunks
    for q in queries:
        if q.kind not in ("question", "lookup") or q.split not in ("dev", "test") or q.user not in directory:
            problems.append(f"{q.id}: unknown kind, split or user")
            continue
        holding = [c for c in chunks if holds(q.evidence, c.text)]
        if q.evidence and not holding:
            problems.append(f"{q.id}: no chunk holds its evidence any more")
        readable = tuple(sorted(c.chunk_id for c in holding if set(c.allowed_groups) & directory[q.user]))
        hidden = tuple(sorted(c.chunk_id for c in holding if not set(c.allowed_groups) & directory[q.user]))
        if readable != q.answer_chunks:
            problems.append(f"{q.id}: labelled {list(q.answer_chunks)}, the evidence is now in {list(readable)}")
        if hidden != q.restricted_chunks:
            problems.append(f"{q.id}: restricted {list(q.restricted_chunks)}, now {list(hidden)}")
        if bool(q.facts) != q.answerable:
            problems.append(f"{q.id}: facts must be given exactly when the question is answerable")
    return problems


documents, _ = ingest(REPO_ROOT)
chunks = [c for d in documents for c in section_chunks(d)]
problems = validate(queries, chunks, manifest)
if file_sha256 != manifest["questions_sha256"]:
    problems.insert(0, f"{QUESTIONS_FILE.name} changed but the manifest didn't: record the change (and a new version)")
print(f"documents now: {len(chunks)} chunks, snapshot {fingerprint(snapshot(chunks))[:12]}... | labels written on "
      f"snapshot {manifest['snapshot']['fingerprint'][:12]}...")
print(f"label problems: {len(problems)}" + "".join(f"\n  {p}" for p in problems))
assert not problems, "The labels don't match the documents: review the lines above before measuring anything."

documents now: 44 chunks, snapshot 3f8344d00b8b... | labels written on snapshot 3f8344d00b8b...
label problems: 0


To see what that catches, here are two ordinary changes, made to a temporary copy of the data: the password policy raises its minimum length from 14 to 16 characters, and the retention policy becomes managers-only.

In [5]:
def update_documents(sample: Path) -> None:
    # Two ordinary changes, made on a temporary copy of the data (Day 05's corpus_after)
    page = sample / "acme-handbook" / "password-and-mfa-standard.md"      # 1. the password rule gets stricter
    text = page.read_text(encoding="utf-8")
    assert "at least 14 characters long" in text
    page.write_text(text.replace("at least 14 characters long", "at least 16 characters long"),
                    encoding="utf-8", newline="\n")
    acl_path = sample / "access-control.json"                             # 2. the retention policy becomes
    acl = json.loads(acl_path.read_text(encoding="utf-8"))                #    managers-only
    acl["permissions"]["acme-handbook/data-retention-policy.md"] = ["managers"]
    acl_path.write_text(json.dumps(acl, indent=2), encoding="utf-8", newline="\n")


stale = validate(queries, corpus_after(REPO_ROOT, update_documents), manifest)
print(f"the same labels after two document changes: {len(stale)} problems")
for p in stale:
    print(f"  {p}")

the same labels after two document changes: 17 problems
  chunk data-retention-policy#deletion-requests: permissions changed
  chunk data-retention-policy#legal-holds: permissions changed
  chunk data-retention-policy#purpose: permissions changed
  chunk data-retention-policy#retention-periods: permissions changed
  chunk password-and-mfa-standard#passwords: text changed
  d03-06: labelled ['data-retention-policy#retention-periods'], the evidence is now in []
  d03-06: restricted [], now ['data-retention-policy#retention-periods']
  d04-v05: no chunk holds its evidence any more
  d04-v05: labelled ['password-and-mfa-standard#passwords'], the evidence is now in []
  d07-a05: labelled ['data-retention-policy#retention-periods'], the evidence is now in []
  d07-a05: restricted [], now ['data-retention-policy#retention-periods']
  d07-a06: labelled ['data-retention-policy#deletion-requests'], the evidence is now in []
  d07-a06: restricted [], now ['data-retention-policy#deletion-requests'

Seventeen problems from two edits. The text change breaks the two labels whose evidence quoted the old rule (a lookup for "14 characters" and today's password question). The permission change is quieter but bigger: four of alice's questions now have their evidence only in chunks she can't read, so their labels must change from "answerable" to "restricted". **Permissions are part of the ground truth.** Nothing here fixes labels automatically: a person decides whether "16 characters" still answers the question, then records a new version of the file and the manifest (LEARN.md, section 8).

## 2. Two retrievers, the same queries, the same snapshot

The two retrievers Day 04 compared, now as each user would use them:

- **BM25** (Day 01's code) has no metadata filter, so each user gets an index of the chunks they may read: 43 for alice, 44 for maria. That is filtering *before* the ranking, the safe order (Day 05).
- **Dense** is Day 06's `retrieve()`: Day 04's model, Day 05's index, the permission filter before the ranking.

Both index the same text (each chunk with its `Title > Section` header). We keep every query's **full** ranking, not just the top 3, so an error can say how far off it was.

In [6]:
embedder = Embedder(REPO_ROOT)     # Day 04's model: downloaded once (133 MB), SHA-256 checked on every run
index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
started = time.perf_counter()
sync(index, chunks, embedder.embed)   # the snapshot's chunks, embedded now (Day 05's sync into an empty index)
rag = Pipeline(index, embedder)       # Day 06's retrieve(): as the user, permission filter before the ranking

# BM25 has no filter: each user gets an index of the chunks they may read (the same as filtering before ranking)
bm25 = {user: BM25Index({c.chunk_id: c.text for c in chunks if set(c.allowed_groups) & groups})
        for user, groups in DIRECTORY.items()}

RETRIEVERS = {
    "BM25": lambda q, k: [(h.id, h.score) for h in bm25[q.user].search(q.question, k)],
    "dense": lambda q, k: [(r.id, r.score) for r in rag.retrieve(q.user, q.question, k)],
}
# Every query's FULL ranking (all chunks the user may read that the retriever scores), so an error can say how far off it was
rankings = {name: {q.id: search(q, len(chunks)) for q in queries} for name, search in RETRIEVERS.items()}
print(f"BM25: one index per user ({', '.join(f'{user} {len(b.ids)} chunks' for user, b in bm25.items())}) | dense: "
      f"{len(index)} vectors, permission filter before the ranking")
print(f"{len(queries)} queries x {len(RETRIEVERS)} retrievers, embedded and ranked in "
      f"{time.perf_counter() - started:.1f} s")

tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified


onnx/model.onnx  133,093,490 bytes   SHA-256 828e1496d7fa... verified


BM25: one index per user (alice 43 chunks, maria 44 chunks) | dense: 44 vectors, permission filter before the ranking
81 queries x 2 retrievers, embedded and ranked in 1.6 s


## 3. Metrics that need labels

For each answerable query, with its labelled chunks:

- **hit@k**: is at least one labelled chunk in the top k? The **hit rate@k** is the share of queries where it is. With k = 3 it answers the question that matters for Day 06's pipeline: did the reader get to see the answer?
- **recall@k**: the share of the labelled chunks that are in the top k. It differs from hit@k only when a query has more than one labelled chunk.
- **reciprocal rank**: 1 / the rank of the first labelled chunk (1 at rank 1, 0.5 at rank 2, …, 0 if it isn't in the top 10). **MRR@10** is its mean: it rewards putting the answer *first*, not just somewhere in the top k.

None of these can be computed without labels. A retriever always returns *something*, with a score; whether that something answers the question is exactly what a label records.

Two worked examples: Day 01's parental-leave question (one labelled chunk), and the lookup "USD 75", which appears in two chunks (the meal rule and the travel table).

In [7]:
def first_relevant(ranked, relevant) -> int | None:
    # The rank of the first labelled chunk, or None if the retriever never returns one
    return next((rank for rank, (cid, _) in enumerate(ranked, start=1) if cid in relevant), None)


def hit(ranked, relevant, k: int) -> bool:          # hit rate@k is the average of this over queries
    rank = first_relevant(ranked, relevant)
    return rank is not None and rank <= k


def recall(ranked, relevant, k: int) -> float:      # recall@k: the share of the labelled chunks found in the top k
    return len(set(relevant) & {cid for cid, _ in ranked[:k]}) / len(relevant)


def reciprocal_rank(ranked, relevant, k: int = MAX_K) -> float:   # MRR@k is the average of this
    rank = first_relevant(ranked, relevant)
    return 1 / rank if rank is not None and rank <= k else 0.0


for qid in ("d03-04", "d04-v25"):
    q = by_qid[qid]
    print(f"{q.id}  \"{q.question}\" ({q.user}); labelled: {', '.join(q.answer_chunks)}")
    for name in RETRIEVERS:
        ranked = rankings[name][q.id]
        top = ", ".join(("*" if cid in q.answer_chunks else "") + cid for cid, _ in ranked[:3])
        print(f"  {name:<6} top 3: {top}")
        print(f"         first relevant at rank {first_relevant(ranked, q.answer_chunks)}: hit@1 "
              f"{hit(ranked, q.answer_chunks, 1):d}, hit@3 {hit(ranked, q.answer_chunks, 3):d}, recall@3 "
              f"{recall(ranked, q.answer_chunks, 3):.2f}, reciprocal rank {reciprocal_rank(ranked, q.answer_chunks):.2f}")
    print()

d03-04  "How long is parental leave?" (alice); labelled: parental-leave-policy#leave-entitlement
  BM25   top 3: parental-leave-policy#who-is-eligible, parental-leave-policy#how-to-request-leave, *parental-leave-policy#leave-entitlement
         first relevant at rank 3: hit@1 0, hit@3 1, recall@3 1.00, reciprocal rank 0.33
  dense  top 3: parental-leave-policy#who-is-eligible, *parental-leave-policy#leave-entitlement, parental-leave-policy#returning-to-work
         first relevant at rank 2: hit@1 0, hit@3 1, recall@3 1.00, reciprocal rank 0.50

d04-v25  "USD 75" (maria); labelled: expense-reimbursement-policy#meals, travel-policy#daily-limits-by-city
  BM25   top 3: *travel-policy#daily-limits-by-city, *expense-reimbursement-policy#meals, expense-reimbursement-policy#approval-and-payment
         first relevant at rank 1: hit@1 1, hit@3 1, recall@3 1.00, reciprocal rank 1.00
  dense  top 3: *expense-reimbursement-policy#meals, expense-reimbursement-policy#approval-and-payment, expens

"Parental leave": BM25 has the answer at rank 3 (reciprocal rank 0.33), dense at rank 2 (0.50); both count as a hit@3. "USD 75": dense puts the Meals chunk first but not the travel table, so hit@3 is 1 and recall@3 is 0.5; BM25 finds both. Which one matters depends on the question: for a lookup, one place that states the value is enough; for a question that needs two passages, recall is the honest number.

Now every query, both retrievers. Unanswerable questions aren't in this table: they have no label to find (section 5 measures them). Each rate comes with a **95% interval**: on 31 questions, one more or one fewer hit moves the rate by 3 points, so the interval says how much the number itself could move with a different sample of questions.

In [8]:
def wilson(successes: int, n: int, z: float = 1.96) -> tuple[float, float]:
    # A 95% interval for a rate measured on n queries (the Wilson score interval): how unsure a small sample is
    p, denominator = successes / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denominator
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
    return centre - half, centre + half


def measure(name: str, selection) -> dict:
    ranked = [(rankings[name][q.id], q.answer_chunks) for q in selection]
    result = {"n": len(ranked)}
    result.update({f"hit@{k}": sum(hit(r, relevant, k) for r, relevant in ranked) for k in K_VALUES})
    result["recall@3"] = sum(recall(r, relevant, 3) for r, relevant in ranked) / len(ranked)
    result["MRR@10"] = sum(reciprocal_rank(r, relevant) for r, relevant in ranked) / len(ranked)
    return result


SELECTIONS = {"questions": [q for q in queries if q.kind == "question" and q.answerable],    # unanswerable ones
              "lookups": [q for q in queries if q.kind == "lookup"]}                         # have no label to find
metrics = {(kind, name): measure(name, selection) for kind, selection in SELECTIONS.items() for name in RETRIEVERS}

print(f"{'':<18}{'n':>3}" + "".join(f"{'hit@' + str(k):>7}" for k in K_VALUES) + f"{'recall@3':>10}{'MRR@10':>8}"
      "   hit@3 rate, 95% interval")
for (kind, name), m in metrics.items():
    low, high = wilson(m["hit@3"], m["n"])
    print(f"{kind + ', ' + name:<18}{m['n']:>3}" + "".join(f"{m['hit@' + str(k)]:>7}" for k in K_VALUES)
          + f"{m['recall@3']:>10.2f}{m['MRR@10']:>8.2f}   {m['hit@3'] / m['n']:.0%} ({low:.0%} to {high:.0%})")
print()
for split in ("dev", "test"):
    part = [q for q in SELECTIONS["questions"] if q.split == split]
    print(f"answerable questions, {split} ({len(part)}): " + " | ".join(
        f"{name} hit@3 {measure(name, part)['hit@3']}, MRR@10 {measure(name, part)['MRR@10']:.2f}" for name in RETRIEVERS))

                    n  hit@1  hit@3  hit@5 hit@10  recall@3  MRR@10   hit@3 rate, 95% interval
questions, BM25    31     24     29     29     30      0.94    0.85   94% (79% to 98%)
questions, dense   31     26     29     31     31      0.94    0.90   94% (79% to 98%)
lookups, BM25      30     30     30     30     30      1.00    1.00   100% (89% to 100%)
lookups, dense     30     12     19     26     29      0.62    0.56   63% (46% to 78%)

answerable questions, dev (16): BM25 hit@3 15, MRR@10 0.85 | dense hit@3 15, MRR@10 0.88
answerable questions, test (15): BM25 hit@3 14, MRR@10 0.84 | dense hit@3 14, MRR@10 0.92


**On questions, the two retrievers are level**: 29 of 31 with the answer in the top 3 each (94%, interval 79% to 98%). Dense puts it first a little more often (26 vs 24 at rank 1, MRR 0.90 vs 0.85) and finds every answer by rank 5. **On lookups they aren't**: BM25 finds all 30 at rank 1; dense finds 19 in the top 3. These are Day 04's numbers exactly (14 of 25 values, 8 at rank 1; all 5 identifiers), which is a useful check in itself: the new harness reproduces an old result.

The intervals overlap heavily on questions. A single metric per retriever can't say *which questions* each one gets right, so compare them **pair by pair**, on the same queries:

In [9]:
def sign_test(only_a: int, only_b: int) -> float:
    # Two-sided exact test on the queries where the two retrievers disagree: if neither were better, each
    # disagreement would be a coin flip. A small p means "unlikely to be luck"; a large p means "can't tell".
    n = only_a + only_b
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(only_a, only_b) + 1)) / 2 ** n)


paired = {}
for kind, selection in SELECTIONS.items():
    found = {name: {q.id for q in selection if hit(rankings[name][q.id], q.answer_chunks, TOP_K)} for name in RETRIEVERS}
    every = {q.id for q in selection}
    paired[kind] = {"both": found["BM25"] & found["dense"], "only BM25": found["BM25"] - found["dense"],
                    "only dense": found["dense"] - found["BM25"], "neither": every - found["BM25"] - found["dense"]}
    p = sign_test(len(paired[kind]["only BM25"]), len(paired[kind]["only dense"]))
    print(f"{kind} ({len(every)}), relevant chunk in the top {TOP_K}: " +
          " | ".join(f"{label} {len(ids)}" for label, ids in paired[kind].items()) + f" | sign test p = {p:.2g}")
    for label in ("only BM25", "only dense", "neither"):
        ids = sorted(paired[kind][label])
        if ids:
            print(f"    {label:<11} " + ", ".join(f"{qid} ({by_qid[qid].question})" for qid in ids))

questions (31), relevant chunk in the top 3: both 28 | only BM25 1 | only dense 1 | neither 1 | sign test p = 1
    only BM25   d03-13 (How soon must I report a lost laptop?)
    only dense  d01-01 (How much time off do new dads get?)
    neither     d07-a08 (Will Acme pay for wine with a business dinner?)
lookups (30), relevant chunk in the top 3: both 19 | only BM25 11 | only dense 0 | neither 0 | sign test p = 0.00098
    only BM25   d04-v06 (14 days), d04-v07 (15 minutes), d04-v10 (3 business days), d04-v11 (30 days), d04-v12 (5 business days), d04-v14 (60 days), d04-v17 (USD 180), d04-v18 (USD 220), d04-v20 (USD 250), d04-v21 (USD 300), d04-v24 (USD 60)


On questions the two retrievers disagree on just two queries, one each way: BM25 finds the lost-laptop answer that dense misses, dense finds the new-dads answer that BM25 never returns. With one disagreement each way the sign test can't tell them apart (p = 1.0). And 30 of 31 questions are answered by at least one of them, which is the case for combining them (Day 08). On lookups BM25 wins all 11 disagreements: if the two were equally good, that would happen about once in a thousand tries (p ≈ 0.001).

Read the counts, not just the percentages: with 31 questions, an "improvement" of one question is a 3-point jump, and it means nothing until you know which question it was and why.

## 4. Every error, with a reason

An **error** here is an answerable query whose labelled chunk isn't in the top 3, the chunks Day 06's reader actually reads. Each one gets a diagnosis, written by hand after reading the ranking and the chunks involved, in four categories:

- **vocabulary**: the question's words aren't the document's words.
- **other section**: the answer sits in a section that never mentions the question's topic.
- **broader chunk**: a general chunk on the same topic outranks the specific one.
- **exact value**: a bare number or amount, which an embedding doesn't pin down.

The `assert` under the table fails if the set of misses ever changes, so a new error can't slip in unexplained and a fixed one can't stay listed.

In [10]:
# Written by hand after reading each miss's ranking and the chunks involved (section 4.1 shows one). The assert
# below fails if the misses change, so a new miss can't go unexplained and a fixed one can't stay listed.
DIAGNOSES = {
    ("BM25", "d01-01"): ("vocabulary", "no word in common with the answer: dads, time off vs caregivers, leave"),
    ("BM25", "d07-a08"): ("vocabulary", "wine vs alcohol, pay for vs reimbursed"),
    ("dense", "d03-13"): ("other section", "'within one hour' is under How to report, which never mentions a laptop"),
    ("dense", "d07-a08"): ("broader chunk", "the policy's Scope (business expenses, meals) outranks the Meals rule"),
    **{("dense", qid): ("exact value", "a bare number or amount: its vector doesn't single out the chunk holding it")
       for qid in ("d04-v06", "d04-v07", "d04-v10", "d04-v11", "d04-v12", "d04-v14", "d04-v17", "d04-v18",
                   "d04-v20", "d04-v21", "d04-v24")},
}

misses = [(name, q) for name in RETRIEVERS for q in queries
          if q.answerable and not hit(rankings[name][q.id], q.answer_chunks, TOP_K)]
assert {(name, q.id) for name, q in misses} == set(DIAGNOSES), "every miss needs a diagnosis, and only misses"

print(f"{len(misses)} misses: an answerable query whose labelled chunk isn't in the top {TOP_K} "
      f"({sum(q.kind == 'question' for _, q in misses)} on questions, {sum(q.kind == 'lookup' for _, q in misses)} "
      "on lookups)\n")
print(f"{'category':<15}{'retriever':<10}{'query':<9}{'labelled chunk':<16}question or lookup")
for name, q in sorted(misses, key=lambda m: (DIAGNOSES[m[0], m[1].id][0], m[0], m[1].id)):
    category, why = DIAGNOSES[name, q.id]
    rank = first_relevant(rankings[name][q.id], q.answer_chunks)
    print(f"{category:<15}{name:<10}{q.id:<9}{'never returned' if rank is None else f'at rank {rank}':<16}"
          f"{q.question}" + (f"\n{'':<50}{why}" if q.kind == "question" else ""))
print(f"\nexact value (all {sum(c == 'exact value' for c, _ in DIAGNOSES.values())}): "
      f"{DIAGNOSES['dense', 'd04-v06'][1]}")
print(" | ".join(f"{category}: {n}" for category, n in Counter(c for c, _ in DIAGNOSES.values()).most_common()))

15 misses: an answerable query whose labelled chunk isn't in the top 3 (4 on questions, 11 on lookups)

category       retriever query    labelled chunk  question or lookup
broader chunk  dense     d07-a08  at rank 4       Will Acme pay for wine with a business dinner?
                                                  the policy's Scope (business expenses, meals) outranks the Meals rule
exact value    dense     d04-v06  at rank 4       14 days
exact value    dense     d04-v07  at rank 19      15 minutes
exact value    dense     d04-v10  at rank 7       3 business days
exact value    dense     d04-v11  at rank 6       30 days
exact value    dense     d04-v12  at rank 4       5 business days
exact value    dense     d04-v14  at rank 9       60 days
exact value    dense     d04-v17  at rank 4       USD 180
exact value    dense     d04-v18  at rank 4       USD 220
exact value    dense     d04-v20  at rank 4       USD 250
exact value    dense     d04-v21  at rank 5       USD 300
exact value

Fifteen errors, and they're not random: **11 of them are dense retrieval on bare values**, the weakness Day 04 found. On natural-language questions there are only four, two per retriever, and they fail for *different reasons*: BM25 on vocabulary (new dads; wine), dense on an answer in a neighbouring section (lost laptop) and on a broader chunk (wine).

Each category points at a technique a later day will **test** (not promise): exact values and vocabulary at hybrid search (Day 08), a broader chunk outranking a specific one at reranking (Day 09), vocabulary at query rewriting (Day 10), an answer in the next section at parent/section retrieval (Day 11). Day 08 measures on these same 81 queries, against the file saved in section 6.

### 4.1 One error, looked at closely

Today's wine question is the one both retrievers miss. Here are both rankings, the words BM25 matched, and the chunk with the answer:

In [11]:
WINE = by_qid["d07-a08"]
print(f"{WINE.id}: {WINE.question}  (labelled: {WINE.answer_chunks[0]})\n")
for name in RETRIEVERS:
    print(name)
    for rank, (cid, score) in enumerate(rankings[name][WINE.id][:6], start=1):
        matched = ""
        if name == "BM25":
            matched = "   matched: " + ", ".join(bm25["alice"].search(WINE.question, 6)[rank - 1].contributions)
        print(f"  #{rank} {score:5.2f}  {cid}{'   <- the answer' if cid in WINE.answer_chunks else ''}{matched}")
meals = next(c for c in chunks if c.chunk_id == WINE.answer_chunks[0])
print(f"\n[{meals.chunk_id}]\n{meals.text}")

d07-a08: Will Acme pay for wine with a business dinner?  (labelled: expense-reimbursement-policy#meals)

BM25
  #1  6.64  expense-reimbursement-policy#scope   matched: acme, pay, business
  #2  3.76  travel-policy#purpose   matched: acme, business
  #3  3.31  parental-leave-policy#returning-to-work   matched: pay
  #4  3.01  data-retention-policy#deletion-requests   matched: acme, business
  #5  2.78  it-software-requests#approval-times   matched: business
  #6  2.21  expense-reimbursement-policy#meals   <- the answer   matched: business
dense
  #1  0.69  expense-reimbursement-policy#scope
  #2  0.66  travel-policy#purpose
  #3  0.61  remote-work-policy#purpose
  #4  0.60  expense-reimbursement-policy#meals   <- the answer
  #5  0.55  vpn-access-guide#installing-the-client
  #6  0.55  remote-work-policy#equipment-and-home-office

[expense-reimbursement-policy#meals]
Expense Reimbursement Policy > Meals
Meals during business travel are reimbursed up to USD 75 per day. Alcohol is not rei

The answer, "Alcohol is not reimbursed", sits in a two-sentence chunk that shares almost no words with the question: BM25 needs *wine*, *pay* or *dinner* to appear, and none do, so it ranks the Meals chunk sixth on "business" alone. Dense comes closer (rank 4) but prefers the policy's Scope chunk, which talks about business expenses and meals in general. A person sees the link (wine is alcohol) at once; neither retriever does.

Before blaming a retriever, **check the label**. Day 03's lost-laptop probe counts only `security-incident-reporting#how-to-report` ("within one hour") as the answer. The travel policy also says to report a lost laptop "straight away": that is a related chunk, not an answer to *how soon*, so it isn't labelled. Decisions like this belong in the `note` field, where the next person can disagree with them.

## 5. Day 06's thresholds, measured

Day 06's pipeline refuses to answer in three places, each set by eye:

| Rule | Day 06 setting | Chosen on |
|---|---|---|
| **The gate**: best retrieval score below this → abstain, no generator called | `MIN_RETRIEVAL_SCORE = 0.50` | one question (France at 0.42) |
| **The names rule**: a name in the question that no source mentions → abstain | (a rule, no number) | three questions (Google, CEO, Wi-Fi) |
| **The reader**: best sentence below this → abstain | `MIN_SENTENCE_SCORE = 0.60` | the 15 answerable questions only |

Now we have 51 labelled questions to measure them on: 31 answerable, 20 not, in both splits. First, what the gate sees: each question's best retrieval score, for the user who asks it.

In [12]:
reader = ExtractiveReader(embedder)       # Day 06's API-free generator, with its default settings


@dataclass(frozen=True)
class Signals:
    top_score: float          # the gate's input: the best retrieval score for this user
    names: tuple[str, ...]    # the names rule: names in the question that none of the sources mentions
    best_piece: float         # the reader's input: its best-scoring sentence or table row


def signals(q: Query) -> Signals:
    results = rag.retrieve(q.user, q.question)
    context = assemble(q.question, results)
    return Signals(results[0].score, tuple(unknown_names(q.question, context.sources)),
                   reader.scored_units(context)[0][0])


natural = [q for q in queries if q.kind == "question"]      # thresholds are for questions, not pasted values
signal = {q.id: signals(q) for q in natural}

print(f"best retrieval score of each question (Day 06's gate: {MIN_RETRIEVAL_SCORE:.2f}) | ● answerable  ○ not answerable\n")
print(f"{'score':<13}{'dev: 16 ● and 8 ○':<26}test: 15 ● and 12 ○")
bins = {q.id: math.floor(signal[q.id].top_score * 50) for q in natural}          # bins of 0.02
for b in range(max(bins.values()), min(bins.values()) - 1, -1):
    row = ["".join("●" for q in natural if q.split == split and q.answerable and bins[q.id] == b) +
           "".join("○" for q in natural if q.split == split and not q.answerable and bins[q.id] == b)
           for split in ("dev", "test")]
    print(f"{b / 50:.2f} to {(b + 1) / 50:.2f} {row[0]:<26}{row[1]}")

best retrieval score of each question (Day 06's gate: 0.50) | ● answerable  ○ not answerable

score        dev: 16 ● and 8 ○         test: 15 ● and 12 ○
0.86 to 0.88                           ●●
0.84 to 0.86 ●                         
0.82 to 0.84 ●                         ●
0.80 to 0.82 ●                         
0.78 to 0.80                           ●●●
0.76 to 0.78 ●●                        ●
0.74 to 0.76 ●●●                       ●
0.72 to 0.74 ●●                        ●●○
0.70 to 0.72 ●●○                       ●●○○
0.68 to 0.70                           ●○
0.66 to 0.68 ●●●                       ●●○
0.64 to 0.66                           
0.62 to 0.64 ●○                        ○
0.60 to 0.62                           ○○
0.58 to 0.60 ○○○                       ○
0.56 to 0.58 ○                         ○○
0.54 to 0.56 ○                         
0.52 to 0.54                           
0.50 to 0.52                           ○
0.48 to 0.50                           
0.46 to 0.48        

Answerable questions (●) sit between 0.63 and 0.88. Unanswerable ones (○) spread from 0.42 to 0.73, **right into the answerable range**: "How long are emails kept?" scores 0.73, higher than 12 of the 31 answerable questions, because the retention policy is about keeping things for a period. A similarity score says how *close* a chunk is to the question, not whether it *answers* it. That is why a metric needs labels, and why no single gate can separate the two groups.

What a gate can do is drop the questions that are clearly off-topic. The rule for choosing it uses only the dev split: *the highest value, in steps of 0.01, that refuses no answerable dev question.* Refusing an answerable question is the mistake we least want for a policy assistant; another product might weigh the two mistakes differently, and should say so. The rule wasn't pre-registered, though: an exploratory run while this lesson was being written had already printed the test split's scores (LEARN.md, section 9.2, has the correction). The test columns below are a check made after that, not a clean held-out result.

In [13]:
def gate_counts(threshold: float, split: str) -> tuple[int, int, int, int]:
    part = [q for q in natural if q.split == split]
    return (sum(q.answerable and signal[q.id].top_score >= threshold for q in part), sum(q.answerable for q in part),
            sum(not q.answerable and signal[q.id].top_score < threshold for q in part), sum(not q.answerable for q in part))


# The rule (dev split only; not pre-registered, see LEARN.md 9.2): the highest gate (in steps of 0.01) that refuses
# no answerable dev question. Refusing an answerable question is the mistake we least want here.
weakest = min((q for q in natural if q.split == "dev" and q.answerable), key=lambda q: signal[q.id].top_score)
MEASURED_GATE = math.floor(signal[weakest.id].top_score * 100) / 100
print(f"weakest answerable dev question: {weakest.id} \"{weakest.question}\" at {signal[weakest.id].top_score:.3f} "
      f"-> measured gate {MEASURED_GATE:.2f}\n")

print(f"{'gate':<6}{'dev: answerable kept':>22}{'not answerable refused':>24}{'test: answerable kept':>24}"
      f"{'not answerable refused':>24}")
for t in sorted({0.50, 0.55, 0.60, MEASURED_GATE, 0.65, 0.70, 0.75}):
    dev, test = gate_counts(t, "dev"), gate_counts(t, "test")
    label = "  <- Day 06" if t == MIN_RETRIEVAL_SCORE else "  <- measured" if t == MEASURED_GATE else ""
    print(f"{t:<6.2f}{f'{dev[0]} of {dev[1]}':>22}{f'{dev[2]} of {dev[3]}':>24}{f'{test[0]} of {test[1]}':>24}"
          f"{f'{test[2]} of {test[3]}':>24}{label}")

top_unanswerable = max((q for q in natural if not q.answerable), key=lambda q: signal[q.id].top_score)
below = sum(q.answerable and signal[q.id].top_score < signal[top_unanswerable.id].top_score for q in natural)
print(f"\nhighest-scoring unanswerable question: {top_unanswerable.id} \"{top_unanswerable.question}\" at "
      f"{signal[top_unanswerable.id].top_score:.3f}, above {below} of the 31 answerable ones")

weakest answerable dev question: d03-11 "What training must new starters complete?" at 0.626 -> measured gate 0.62

gate    dev: answerable kept  not answerable refused   test: answerable kept  not answerable refused
0.50                16 of 16                  1 of 8                15 of 15                 0 of 12  <- Day 06
0.55                16 of 16                  1 of 8                15 of 15                 1 of 12
0.60                16 of 16                  6 of 8                15 of 15                 4 of 12
0.62                16 of 16                  6 of 8                15 of 15                 6 of 12  <- measured
0.65                15 of 16                  7 of 8                15 of 15                 7 of 12
0.70                12 of 16                  7 of 8                12 of 15                 9 of 12
0.75                 6 of 16                  8 of 8                 8 of 15                12 of 12

highest-scoring unanswerable question: d07-u03 "How

**Day 06's gate at 0.50 was safe and nearly useless**: it refused one unanswerable question out of 20 (France). The measured gate, **0.62**, still keeps every answerable question in both splits, and refuses 6 of 8 unanswerable dev questions and **6 of 12 on the test split**, which it was not chosen on. That last number is the honest one.

It is also fragile in a way worth stating: the gate is set by a single question, the weakest answerable one ("What training must new starters complete?", at 0.626). One new question below it would move the gate, which is how it should be, and why the set has to keep growing.

### 5.1 The names rule

Day 06 added a rule: if the question names something (a capitalised word, after the first) that no source mentions, abstain. On the questions it was written for, it looked perfect:

In [14]:
for split in ("dev", "test"):
    part = [q for q in natural if q.split == split]
    caught = [q for q in part if not q.answerable and signal[q.id].names]
    wrong = [q for q in part if q.answerable and signal[q.id].names]
    print(f"{split}: refuses {len(caught)} of {sum(not q.answerable for q in part)} unanswerable questions "
          f"({', '.join(n for q in caught for n in signal[q.id].names) or 'none'}) and {len(wrong)} of "
          f"{sum(q.answerable for q in part)} answerable ones")
    for q in wrong:
        print(f"    {q.id} \"{q.question}\": no source mentions {', '.join(signal[q.id].names)}")

dev: refuses 4 of 8 unanswerable questions (Google, France, CEO, Wi-Fi) and 0 of 16 answerable ones
test: refuses 0 of 12 unanswerable questions (none) and 2 of 15 answerable ones
    d07-a01 "What is the hotel limit per night in Tokyo?": no source mentions Tokyo
    d07-a11 "My laptop might have a virus. Should I switch it off?": no source mentions Should


**On dev it flags four unanswerable questions and no answerable one** (France never gets this far: the gate refuses it first). **On test it flags no unanswerable question and two answerable ones.** "Tokyo" isn't in any source, but the travel policy answers the question anyway: "Use the Other cities row for any city not listed". And "Should" is the first word of the question's *second sentence*: the rule only skips the first word of the question, so any capitalised word after a full stop counts as a name. Neither problem could show up on the dev questions, which are all one sentence and name only things the documents don't cover.

This is what a held-out split is for: **a rule can be 100% right on the questions that inspired it.** Day 07 doesn't fix the rule (that changes Day 06's code); it records both false refusals as known failures (LEARN.md, section 10).

### 5.2 The reader's threshold

The reader's threshold applies to questions that get past the gate and the names rule. Same rule for choosing it, on the dev split:

In [15]:
# The reader's threshold, with the measured gate in front of it: only questions past the gate and the names rule
reaching = [q for q in natural if signal[q.id].top_score >= MEASURED_GATE and not signal[q.id].names]
weakest_piece = min((q for q in reaching if q.split == "dev" and q.answerable), key=lambda q: signal[q.id].best_piece)
MEASURED_READER = math.floor(signal[weakest_piece.id].best_piece * 100) / 100     # the same rule as the gate
print(f"{len(reaching)} of {len(natural)} questions get past the measured gate and the names rule")
print(f"weakest answerable dev piece: {weakest_piece.id} \"{weakest_piece.question}\" at "
      f"{signal[weakest_piece.id].best_piece:.3f} -> measured reader threshold {MEASURED_READER:.2f}\n")
print(f"{'threshold':<11}{'dev: answerable kept':>22}{'not answerable refused':>24}{'test: answerable kept':>24}"
      f"{'not answerable refused':>24}")
for t in sorted({MIN_SENTENCE_SCORE, MEASURED_READER, 0.65, 0.70}):
    cells = []
    for split in ("dev", "test"):
        part = [q for q in reaching if q.split == split]
        cells += [f"{sum(q.answerable and signal[q.id].best_piece >= t for q in part)} of {sum(q.answerable for q in part)}",
                  f"{sum(not q.answerable and signal[q.id].best_piece < t for q in part)} of "
                  f"{sum(not q.answerable for q in part)}"]
    label = "  <- Day 06" if t == MIN_SENTENCE_SCORE else "  <- measured" if t == MEASURED_READER else ""
    print(f"{t:<11.2f}{cells[0]:>22}{cells[1]:>24}{cells[2]:>24}{cells[3]:>24}{label}")

35 of 51 questions get past the measured gate and the names rule
weakest answerable dev piece: d03-11 "What training must new starters complete?" at 0.639 -> measured reader threshold 0.63

threshold    dev: answerable kept  not answerable refused   test: answerable kept  not answerable refused
0.60                     16 of 16                  0 of 0                13 of 13                  0 of 6  <- Day 06
0.63                     16 of 16                  0 of 0                13 of 13                  1 of 6  <- measured
0.65                     15 of 16                  0 of 0                13 of 13                  1 of 6
0.70                     14 of 16                  0 of 0                12 of 13                  2 of 6


With the measured gate in front, **no unanswerable dev question reaches the reader at all** (0 of 0): the dev split can only say how low the threshold must stay to keep every answer. The weakest answerable dev question sets it again ("new starters", best piece 0.639), so the measured value is **0.63**. It adds little beyond the measured gate: on the test split it refuses one more unanswerable question (the company phone), and nothing answerable.

### 5.3 End to end: Day 06's settings vs the measured ones

Finally, the whole Day 06 pipeline with each pair of settings, graded the simple way Day 06 did: an answer is *correct* if it contains the labelled facts, *refused* if the user sees no answer (abstained, or blocked by the citation check).

In [16]:
def outcome(a, q: Query) -> str:
    # What the user experiences. "refused": no answer shown (abstained, or blocked by the citation check)
    if a.status != "answered":
        return "refused"
    if not q.answerable:
        return "answered"            # an answer to a question the documents don't answer for this user
    return "correct" if all(f.lower() in a.text.lower() for f in q.facts) else "wrong"


SETTINGS = {f"Day 06 ({MIN_RETRIEVAL_SCORE:.2f} / {MIN_SENTENCE_SCORE:.2f})": (MIN_RETRIEVAL_SCORE, MIN_SENTENCE_SCORE),
            f"measured ({MEASURED_GATE:.2f} / {MEASURED_READER:.2f})": (MEASURED_GATE, MEASURED_READER)}
answers, outcomes = {}, {}
for label, (gate, piece) in SETTINGS.items():
    pipeline = Pipeline(index, embedder, min_retrieval_score=gate)
    generator = ExtractiveReader(embedder, min_score=piece)
    answers[label] = {q.id: pipeline.answer(q.user, q.question, generator) for q in natural}
    outcomes[label] = {q.id: outcome(answers[label][q.id], q) for q in natural}

print(f"{'settings (gate / reader)':<26}{'split':<7}{'answerable: correct / wrong / refused':<40}"
      "not answerable: refused / answered")
for label in SETTINGS:
    for split in ("dev", "test"):
        part = [q for q in natural if q.split == split]
        a = Counter(outcomes[label][q.id] for q in part if q.answerable)
        u = Counter(outcomes[label][q.id] for q in part if not q.answerable)
        answerable = f"{a['correct']} / {a['wrong']} / {a['refused']}  (of {sum(a.values())})"
        print(f"{label:<26}{split:<7}{answerable:<40}{u['refused']} / {u['answered']}  (of {sum(u.values())})")

day06, measured = SETTINGS
changed = [q for q in natural if outcomes[day06][q.id] != outcomes[measured][q.id]]
print(f"\nquestions whose outcome changed: {len(changed)}")
for q in changed:
    print(f"  {q.id:<9}{q.split:<6}{q.question:<55}{outcomes[day06][q.id]} -> {outcomes[measured][q.id]}")
still = [q for q in natural if q.split == "test" and outcomes[measured][q.id] == "answered"]
print(f"\nunanswerable test questions still answered with the measured settings: {len(still)}")
for q in still:
    a = answers[measured][q.id]
    print(f"  {q.id} {q.question}\n      -> {a.text[:110]}{'...' if len(a.text) > 110 else ''}")
blocked = [q for q in natural if answers[day06][q.id].status == "blocked"]
for q in blocked:
    a = answers[day06][q.id]
    print(f"\nblocked by Day 06's citation check: {q.id} \"{q.question}\"\n  reply:    {a.reply.text}\n"
          f"  problems: {a.check.problems} (the checker split \"{a.check.claims[0].text}\" off as a sentence)")

settings (gate / reader)  split  answerable: correct / wrong / refused   not answerable: refused / answered
Day 06 (0.50 / 0.60)      dev    14 / 2 / 0  (of 16)                     6 / 2  (of 8)
Day 06 (0.50 / 0.60)      test   12 / 1 / 2  (of 15)                     5 / 7  (of 12)
measured (0.62 / 0.63)    dev    14 / 2 / 0  (of 16)                     8 / 0  (of 8)
measured (0.62 / 0.63)    test   12 / 1 / 2  (of 15)                     7 / 5  (of 12)

questions whose outcome changed: 4
  d06-02   dev   When do pay rises take effect?                         answered -> refused
  d06-05   dev   How do I book a meeting room?                          answered -> refused
  d07-u02  test  How many sick days can I take?                         answered -> refused
  d07-u10  test  How do I enrol in health insurance?                    answered -> refused

unanswerable test questions still answered with the measured settings: 5
  d07-u01 How much annual leave do I get?
      -> Primary careg

On the dev split, the measured settings refuse all eight unanswerable questions, but that is the split they were tuned on, so it proves little. On the **test split**: still 12 correct answers out of 15, and 7 of 12 unanswerable questions refused instead of 5, with no answerable question lost.

Three more things this run shows:

- **Five unanswerable test questions still get an answer**, a cited sentence on the right topic that doesn't answer the question (annual leave gets the parental-leave entitlement). Their scores are as high as answerable questions' scores; no threshold on these numbers separates them. That takes something that reads the question and the passage together: a reranker (Day 09) or a language model, measured (Day 12).
- **The test split found a Day 06 bug.** For the company-phone question the reader copied a numbered list item ("3. If it is not listed…"), and the citation checker split "3." off as a sentence without a citation, so the answer was blocked. It failed closed (the answer was wrong anyway), but a correct numbered step would have been blocked too.
- **Wrong answers didn't change**: the wine question is answered wrongly with either setting, because the answer isn't in the top 3 (section 4). Thresholds can't fix retrieval.

Day 07 doesn't change Day 06's defaults; `answering.py` keeps them, so Day 06's notebook and the module still agree. It records the measured values (0.62 and 0.63) in the baseline file, for later days to adopt and re-measure.

## 6. Save the baseline

A baseline is only useful if the next change is compared with *it*, not with someone's memory of it. The notebook writes this run to `outputs/day-07/retrieval-baseline.json` (git-ignored): the dataset version and snapshot, both retrievers' settings, every metric, every query's rank and top 3, every error with its diagnosis, and the measured thresholds with their outcomes. **The accepted copy is committed** as [`evals/baselines/day-07-retrieval.json`](../evals/baselines/day-07-retrieval.json), and every run is compared with it: ranks and counts exactly, scores within 0.005.

Changing the accepted baseline is a deliberate act: someone reviews the differences, copies the new file over the old one, and commits it with a reason. Day 08 will add hybrid search and run this comparison.

In [17]:
def top_score(name: str, qid: str) -> float | None:
    ranked = rankings[name][qid]
    return round(ranked[0][1], 3) if ranked else None


def tie_margin(name: str, q: Query) -> float | None:
    # How far the first labelled chunk's score is from its neighbours' in the ranking. A tiny margin is a near
    # tie: another CPU's rounding could swap the two, which is not a change in the retriever.
    ranked, rank = rankings[name][q.id], first_relevant(rankings[name][q.id], q.answer_chunks)
    if rank is None:
        return None
    gaps = [ranked[i - 1][1] - ranked[i][1] for i in (rank - 1, rank) if 0 < i < len(ranked)]
    return round(min(gaps), 4) if gaps else None


baseline = {
    "name": "Day 07 retrieval baseline",
    "dataset": {"questions": QUESTIONS_FILE.relative_to(REPO_ROOT).as_posix(), "version": manifest["version"],
                "sha256": file_sha256, "snapshot": manifest["snapshot"]["fingerprint"]},
    "retrievers": {
        "BM25": {"code": "rag_from_scratch.bm25 (Day 01)", "k1": bm25["alice"].k1, "b": bm25["alice"].b,
                 "indexed_text": "chunk text with its Title > Section header",
                 "permissions": "one index of each user's readable chunks (filter before the ranking)"},
        "dense": {"code": "rag_from_scratch.answering.Pipeline.retrieve (Day 06)", "model": embedder.model["repo"],
                  "revision": embedder.revision, "query_instruction": True,
                  "indexed_text": "chunk text with its Title > Section header",
                  "permissions": "metadata filter before the ranking"}},
    "top_k": TOP_K,
    "metrics": {f"{kind}, {name}": {k: round(v, 4) if isinstance(v, float) else v for k, v in m.items()}
                for (kind, name), m in metrics.items()},
    "queries": {q.id: {name: {"first_relevant": first_relevant(rankings[name][q.id], q.answer_chunks) if q.answerable
                              else None,
                              "margin": tie_margin(name, q) if q.answerable else None,
                              "top3": [cid for cid, _ in rankings[name][q.id][:TOP_K]],
                              "top_score": top_score(name, q.id)}
                       for name in RETRIEVERS} for q in queries},
    "errors": [{"retriever": name, "id": q.id, "first_relevant": first_relevant(rankings[name][q.id], q.answer_chunks),
                "category": DIAGNOSES[name, q.id][0], "why": DIAGNOSES[name, q.id][1]} for name, q in misses],
    "thresholds": {
        "rule": "the highest value (in steps of 0.01) that refuses no answerable dev question",
        "day06": {"min_retrieval_score": MIN_RETRIEVAL_SCORE, "min_sentence_score": MIN_SENTENCE_SCORE},
        "measured": {"min_retrieval_score": MEASURED_GATE, "min_sentence_score": MEASURED_READER},
        "outcomes": {label: {split: dict(sorted(Counter(outcomes[label][q.id] + (" (answerable)" if q.answerable else
                                                                                   " (not answerable)")
                                                        for q in natural if q.split == split).items()))
                             for split in ("dev", "test")} for label in SETTINGS}},
}


TIE = 0.001       # scores closer than this are a tie in practice (CPUs differ around the 7th decimal)


def compare(new: dict, old: dict, tolerance: float = 0.005) -> tuple[list[str], list[str]]:
    # What changed since the accepted baseline: (changes, near ties). Ranks and counts must match exactly, scores
    # within a tolerance. A rank that moves where the baseline recorded a near tie is listed apart, not as a change.
    if new["dataset"] != old["dataset"]:
        return ["different labels or a different snapshot: these two runs can't be compared"], []
    changes = ["retriever settings changed"] if new["retrievers"] != old["retrievers"] else []
    ties = []
    for qid, per_retriever in new["queries"].items():
        for name, now in per_retriever.items():
            then = old["queries"].get(qid, {}).get(name)
            if then is None:
                changes.append(f"{qid} {name}: not in the baseline")
            elif now["first_relevant"] != then["first_relevant"]:
                moved = f"{qid} {name}: labelled chunk at rank {then['first_relevant']} -> {now['first_relevant']}"
                if then["margin"] is not None and then["margin"] < TIE:
                    ties.append(f"{moved} (a near tie: {then['margin']} from its neighbour)")
                else:
                    changes.append(moved)
            elif (now["top_score"] is None) != (then["top_score"] is None) or \
                    (now["top_score"] is not None and abs(now["top_score"] - then["top_score"]) > tolerance):
                changes.append(f"{qid} {name}: top score {then['top_score']} -> {now['top_score']}")
    moved_metrics = [f"{key}: {old['metrics'].get(key)} -> {m}" for key, m in new["metrics"].items()
                     if m != old["metrics"].get(key)]
    (ties if ties and not changes else changes).extend(moved_metrics)   # metrics follow from the ranks
    if new["thresholds"] != old["thresholds"]:
        changes.append("measured thresholds or their outcomes changed")
    return changes, ties


OUT_DIR.mkdir(parents=True, exist_ok=True)
run_file = OUT_DIR / "retrieval-baseline.json"
run_file.write_text(json.dumps(baseline, indent=1, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
print(f"this run: {run_file.relative_to(REPO_ROOT).as_posix()} ({len(baseline['queries'])} queries, "
      f"{len(baseline['errors'])} errors)")
if BASELINE_FILE.is_file():
    changes, ties = compare(baseline, json.loads(BASELINE_FILE.read_text(encoding="utf-8")))
    print(f"compared with {BASELINE_FILE.relative_to(REPO_ROOT).as_posix()}: "
          + (f"{len(changes)} changes" + "".join(f"\n  {c}" for c in changes) if changes else
             f"no changes ({len(queries)} queries x {len(RETRIEVERS)} retrievers, metrics and thresholds)"))
    for t in ties:
        print(f"  near tie, not a change: {t}")
    fragile = sorted((entry["margin"], qid, name) for qid, per in baseline["queries"].items()
                     for name, entry in per.items() if entry["margin"] is not None and entry["margin"] < 0.002)
    print("closest calls in this run: " + ", ".join(f"{qid} {name} ({m})" for m, qid, name in fragile))
else:
    changes, ties = None, []
    print(f"no accepted baseline yet: review {run_file.name}, then copy it to "
          f"{BASELINE_FILE.relative_to(REPO_ROOT).as_posix()} and commit it")

this run: outputs/day-07/retrieval-baseline.json (81 queries, 15 errors)
compared with evals/baselines/day-07-retrieval.json: no changes (81 queries x 2 retrievers, metrics and thresholds)
closest calls in this run: d03-13 dense (0.0001), d04-i03 dense (0.0008), d04-v10 dense (0.0009), d04-v21 dense (0.0017)


No changes: this run reproduces the accepted baseline. The last line lists the closest calls, rankings where the labelled chunk's score is less than 0.002 from its neighbour's. That is still far more than runs differ by (about 0.0000001 here, from the order of the arithmetic), but if another CPU ever swaps one of them, the comparison reports it as a **near tie**, not as a change.

If you change something (exercise 3), this cell lists exactly which queries moved, and the sanity check below fails until you either undo the change or accept the new baseline on purpose.

## 7. Sanity checks

Each check pins a result quoted in this notebook or in LEARN.md. Similarity scores are compared with a tolerance (CPUs differ in the last decimal places); ids, ranks and counts exactly.

In [18]:
def close(x: float, expected: float, tol: float = 0.01) -> bool:
    return abs(x - expected) <= tol


def ranks(name: str, ids) -> list:
    return [first_relevant(rankings[name][qid], by_qid[qid].answer_chunks) for qid in ids]


# The labelled set (section 1)
assert len(queries) == 81 and manifest["counts"] == {"queries": 81, "questions": 51, "answerable_questions": 31,
                                                     "unanswerable_questions": 20, "lookups": 30, "dev": 54, "test": 27}
assert sum(q.answerable for q in natural) == 31 and all(q.answerable for q in queries if q.kind == "lookup")
assert all(q.split == "test" for q in queries if q.origin == "day07-new") and problems == []
assert by_qid["d06-02"].restricted_chunks == ("manager-pay-review-guide#overview",) and not by_qid["d06-02"].answerable
assert len(stale) == 17 and sum(p.startswith("chunk ") for p in stale) == 5
assert "d07-a10: no chunk holds its evidence any more" in stale and any(p.startswith("d03-06: labelled") for p in stale)

# Two retrievers (sections 2 and 3)
assert {u: len(b.ids) for u, b in bm25.items()} == {"alice": 43, "maria": 44} and len(index) == 44
assert {key: (m["hit@1"], m["hit@3"], m["hit@5"], m["hit@10"]) for key, m in metrics.items()} == {
    ("questions", "BM25"): (24, 29, 29, 30), ("questions", "dense"): (26, 29, 31, 31),
    ("lookups", "BM25"): (30, 30, 30, 30), ("lookups", "dense"): (12, 19, 26, 29)}
# (d04-i03 is 0.0008 below rank 1 and d03-13 0.0001 above rank 5: still far above the ~1e-7 differences between runs)
assert close(metrics["questions", "BM25"]["MRR@10"], 0.849, 0.001) and close(metrics["questions", "dense"]["MRR@10"], 0.898, 0.003)
assert close(metrics["lookups", "dense"]["recall@3"], 18.5 / 30, 1e-9) and metrics["lookups", "BM25"]["MRR@10"] == 1.0
assert ranks("BM25", ["d03-04", "d03-13", "d01-01", "d07-a08"]) == [3, 3, None, 6]
assert ranks("dense", ["d03-04", "d01-01", "d07-a08", "d04-v07"]) == [2, 3, 4, 19]
assert ranks("dense", ["d03-13"])[0] in (4, 5)              # a near tie with rank 5 (0.0001 apart)
assert recall(rankings["dense"]["d04-v25"], by_qid["d04-v25"].answer_chunks, 3) == 0.5      # USD 75: one of two
assert recall(rankings["BM25"]["d04-v25"], by_qid["d04-v25"].answer_chunks, 3) == 1.0
# Day 04 measured the lookups without users; maria reads all 44 chunks, so the numbers must be Day 04's
values = [q for q in queries if "exact-value" in q.tags]
assert sum(hit(rankings["dense"][q.id], q.answer_chunks, 3) for q in values) == 14
assert sum(hit(rankings["dense"][q.id], q.answer_chunks, 1) for q in values) == 8
assert {k: len(v) for k, v in paired["questions"].items()} == {"both": 28, "only BM25": 1, "only dense": 1, "neither": 1}
assert paired["questions"]["only BM25"] == {"d03-13"} and paired["questions"]["only dense"] == {"d01-01"}
assert {k: len(v) for k, v in paired["lookups"].items()} == {"both": 19, "only BM25": 11, "only dense": 0, "neither": 0}
assert sign_test(1, 1) == 1.0 and sign_test(11, 0) < 0.001
assert close(wilson(29, 31)[0], 0.793, 0.001) and close(wilson(29, 31)[1], 0.982, 0.001) and wilson(30, 30)[1] == 1.0
# A permission regression check: no restricted chunk in any user's top 10, for either retriever
assert not any(set(q.restricted_chunks) & {cid for cid, _ in rankings[name][q.id][:MAX_K]}
               for q in queries for name in RETRIEVERS)

# Every error (section 4)
assert len(misses) == 15 and Counter(c for c, _ in DIAGNOSES.values()) == {
    "exact value": 11, "vocabulary": 2, "other section": 1, "broader chunk": 1}

# Thresholds (section 5)
assert weakest.id == "d03-11" and close(signal["d03-11"].top_score, 0.626, 0.002) and MEASURED_GATE == 0.62
assert gate_counts(0.50, "dev") == (16, 16, 1, 8) and gate_counts(0.50, "test") == (15, 15, 0, 12)
assert gate_counts(MEASURED_GATE, "dev") == (16, 16, 6, 8) and gate_counts(MEASURED_GATE, "test") == (15, 15, 6, 12)
assert top_unanswerable.id == "d07-u03" and below == 12
assert {q.id: signal[q.id].names for q in natural if signal[q.id].names} == {
    "d01-03": ("Google",), "d01-04": ("France",), "d06-04": ("CEO",), "d06-06": ("Wi-Fi",),
    "d07-a01": ("Tokyo",), "d07-a11": ("Should",)}
assert weakest_piece.id == "d03-11" and MEASURED_READER == 0.63 and len(reaching) == 35
assert not any(not q.answerable for q in reaching if q.split == "dev")    # the gate leaves the reader no dev ○
for label, expected in zip(SETTINGS, [{"dev": ((14, 2, 0), (6, 2)), "test": ((12, 1, 2), (5, 7))},
                                      {"dev": ((14, 2, 0), (8, 0)), "test": ((12, 1, 2), (7, 5))}]):
    for split, (a, u) in expected.items():
        part = [q for q in natural if q.split == split]
        got_a = Counter(outcomes[label][q.id] for q in part if q.answerable)
        got_u = Counter(outcomes[label][q.id] for q in part if not q.answerable)
        assert (got_a["correct"], got_a["wrong"], got_a["refused"]) == a, (label, split, got_a)
        assert (got_u["refused"], got_u["answered"]) == u, (label, split, got_u)
assert [q.id for q in blocked] == ["d07-u05"] and answers[day06]["d07-u05"].check.problems == ["no citation"]
assert {q.id for q in still} == {"d07-u01", "d07-u03", "d07-u04", "d07-u11", "d07-u12"}

# The baseline file (section 6)
saved = json.loads(run_file.read_text(encoding="utf-8"))
assert saved["dataset"]["snapshot"] == fingerprint(snapshot(chunks)) and len(saved["errors"]) == 15
assert changes is None or changes == [], changes            # near ties (CPU rounding) are allowed

print("All Day 07 sanity checks passed.")

All Day 07 sanity checks passed.


## 8. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing anything.

1. **Label three questions of your own** (one answerable, one only maria can answer, one the documents don't answer). Build each as a `Query` with your evidence and empty `answer_chunks` and `restricted_chunks`, and run `validate([...], chunks, manifest)` on just those: the problems it lists are the labels your evidence implies. Fill them in until `validate` finds nothing, then see where each retriever ranks them with `RETRIEVERS["dense"](q, 10)`.
2. **Make a label stale on purpose.** In `update_documents`, change the travel policy's permissions instead (`acme-handbook/travel-policy.pdf`). Which labels break, and why do the lookups (asked by maria) survive?
3. **Change a retriever and read the comparison.** Index BM25 without the `Title > Section` header (`c.text.split("\n", 1)[1]`), rerun, and read section 6's list of changes. Which queries moved, and does Day 04's finding (headers help BM25) hold on 81 queries? The sanity checks fail until you undo the change: that is the comparison doing its job.
4. **Choose the gate with another rule.** Instead of "refuse no answerable dev question", allow one false refusal on dev. What gate does that give, and what happens on the test split? Write down which mistake you'd rather make for a pay or security question, and why.
5. **Fix the names rule** in a copy of `unknown_names()` so that it skips the first word of *every* sentence of the question. How many test questions change outcome? Is "Tokyo" fixable by a rule at all?